In [1]:
import pprint
import radiate as rd
import numpy as np
import polars as pl

rd.random.seed(123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [3]:
collector = rd.MetricCollector()
engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alters(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(rd.Limit.score(0.01), rd.Limit.generations(500))
)

result = engine.run(log=True)

2026-04-09T23:22:10.016571Z  INFO Epoch 1    | Score:   1.1841 | Time: 2.43ms
2026-04-09T23:22:10.017822Z  INFO Epoch 2    | Score:   0.9343 | Time: 3.60ms
2026-04-09T23:22:10.018827Z  INFO Epoch 3    | Score:   0.8793 | Time: 4.55ms
2026-04-09T23:22:10.020146Z  INFO Epoch 4    | Score:   0.8793 | Time: 5.80ms
2026-04-09T23:22:10.021234Z  INFO Epoch 5    | Score:   0.8697 | Time: 6.81ms
2026-04-09T23:22:10.022163Z  INFO Epoch 6    | Score:   0.7182 | Time: 7.69ms
2026-04-09T23:22:10.023047Z  INFO Epoch 7    | Score:   0.7182 | Time: 8.55ms
2026-04-09T23:22:10.023928Z  INFO Epoch 8    | Score:   0.5674 | Time: 9.39ms
2026-04-09T23:22:10.025003Z  INFO Epoch 9    | Score:   0.5674 | Time: 10.43ms
2026-04-09T23:22:10.025911Z  INFO Epoch 10   | Score:   0.5674 | Time: 11.30ms
2026-04-09T23:22:10.027066Z  INFO Epoch 11   | Score:   0.5085 | Time: 12.37ms
2026-04-09T23:22:10.028077Z  INFO Epoch 12   | Score:   0.3981 | Time: 13.31ms
2026-04-09T23:22:10.029144Z  INFO Epoch 13   | Score:   0.37

In [4]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""carryover_rate""",0.175258,81.924034,0.163848,0.018933,0.000358,0.0,0.0,0.2,500,null,null,null,null,null,null,500,1,"[""derived"", ""statistic"", ""rate""]"
"""lineage_parents_ratio""",0.56,254.380005,0.50876,0.054842,0.003008,0.0,0.36,0.65,500,null,null,null,null,null,null,500,1,"[""age"", ""statistic"", ""lineage""]"
"""roulette_selector""",20.0,10000.0,20.0,0.0,0.0,0.0,20.0,20.0,500,2200µs,4µs,2µs,3µs,68µs,0µs,500,2,"[""selector"", ""statistic"", ""time""]"
"""blend_crossover""",1013.0,579155.0,1158.310059,148.555573,22068.759766,0.140574,752.0,1570.0,500,8715µs,17µs,3µs,12µs,56µs,0µs,500,2,"[""alterer"", ""crossover"", … ""time""]"
"""alter_cross_family""",43.0,27071.0,44.671616,20.194004,407.797791,-1.554685,1.0,64.0,606,null,null,null,null,null,null,500,1,"[""alterer"", ""statistic"", … ""lineage""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""filter_step""",0.0,null,null,null,null,null,null,null,500,3240µs,6µs,0µs,6µs,19µs,0µs,500,1,"[""time"", ""step""]"
"""audit_step""",0.0,null,null,null,null,null,null,null,500,9014µs,18µs,18µs,12µs,368µs,0µs,500,1,"[""time"", ""step""]"
"""alter_parent_reuse""",1.0,67071.0,2.636646,4.82692,23.29915,5.779857,1.0,58.0,25438,null,null,null,null,null,null,500,56,"[""alterer"", ""statistic"", … ""lineage""]"


In [5]:
print(result.metrics().dashboard())
# pprint.pprint(metrics["carryover_rate"].tags())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["carryover_rate"].to_dict())

[34 metrics, 16021 updates]
----- Metrics -----
  carryover: 0.164  diversity: 0.958  unique_members: 97  unique_scores: 97  improvements: 17  iter_time(mean): 871.284µs

== Statistics ==
Name                     | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                      | value  | 0.426      | 0.000      | 20.000     | 50000  | -            | 1.390      | 19.663     | 263.791    | -         
alter_cross_family       | value  | 44.672     | 1.000      | 64.000     | 606    | -            | 20.194     | -1.555     | 3.655      | -         
alter_parent_reuse       | value  | 2.637      | 1.000      | 58.000     | 25438  | -            | 4.827      | 5.780      | 41.011     | -         
alter_within_family      | value  | 105.292    | 1.000      | 144.000 

In [6]:
df = collector.to_polars(lazy=False)

df = (
    df.filter(
        (pl.col("update_count") > 1) & (~pl.col("tags").list.contains("distribution"))
    )
    .select("name", "version", "update_count", "tags")
    .group_by("name")
    .agg(
        pl.col("update_count").sum().alias("update_count"),
    )
    .sort("update_count", descending=False)
)

df

name,update_count
str,i64
"""evaluation_count""",2
"""blend_crossover""",998
"""gaussian_mutator""",998
"""roulette_selector""",1000
"""boltzmann_selector""",1000
"""evaluate_step""",1000


In [7]:
df = collector.to_polars(lazy=False)


In [8]:
df.filter(pl.col("name") == "gaussian_mutator")

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""gaussian_mutator""",2115.0,2115.0,2115.0,0.0,0.0,NaN,2115.0,2115.0,1,87µs,87µs,0µs,87µs,87µs,0µs,1,0,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2083.0,4198.0,2099.0,22.627417,512.0,NaN,2083.0,2115.0,2,172µs,86µs,2µs,84µs,87µs,0µs,2,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2100.0,6298.0,2099.333252,16.010414,256.333344,-0.346677,2083.0,2115.0,3,249µs,83µs,5µs,76µs,87µs,0µs,3,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2086.0,8384.0,2096.0,14.674222,215.332779,1.221628,2083.0,2115.0,4,320µs,80µs,7µs,71µs,87µs,0µs,4,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2086.0,10470.0,2094.0,13.472178,181.499588,1.713732,2083.0,2115.0,5,399µs,79µs,6µs,71µs,87µs,0µs,5,2,"[""alterer"", ""mutator"", … ""time""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""gaussian_mutator""",2129.0,1.047143e6,2111.175293,37.73447,1423.890259,-0.073169,1998.0,2210.0,496,35496µs,71µs,3µs,67µs,104µs,0µs,496,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2091.0,1.049234e6,2111.134766,37.707275,1421.838623,-0.070234,1998.0,2210.0,497,35566µs,71µs,3µs,67µs,104µs,0µs,497,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2016.0,1.05125e6,2110.943848,37.909782,1437.151611,-0.08579,1998.0,2210.0,498,35635µs,71µs,3µs,67µs,104µs,0µs,498,2,"[""alterer"", ""mutator"", … ""time""]"
